In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# Change to project root directory (parent of scripts folder)
current_dir = Path.cwd()
if current_dir.name == 'scripts':
    os.chdir(current_dir.parent)


In [ ]:
# Split documents into smaller chunks for processing and indexing
from lib.data.chunker import NaiveChunker

chunker = NaiveChunker(
    input_dir="./data_lzj/documents/",
    output_dir="./data_lzj/chunks/",
    chunk_size=500,
    chunk_overlap=100,
    reload=False
)
chunker.run()

In [ ]:
# Extract titles from documents for title-based search indexing
from lib.data.title_extractor import TitleExtractor

extractor = TitleExtractor(
    input_dir="./data_lzj/documents/",
    output_file="./data_lzj/titles.json"
)
extractor.run()

In [ ]:
# LLM Generate summaries for documents using LLM to create concise representations
# cost $1
from lib.data.summary_extractor import SummaryExtractor
import dotenv
dotenv.load_dotenv()

extractor = SummaryExtractor(
    input_dir="./data_lzj/documents/",
    output_file="./data_lzj/summaries.json",
    max_workers=8,
    limit=None
)
await extractor.run(skip_existing=True)

In [ ]:
# Initialize Elasticsearch client for indexing and searching document chunks
from lib.search.elastic_chunk_index import ElasticWriteClientChunks
elastic_chunk = ElasticWriteClientChunks(
    chunk_index_name="lzj",
    chunks_path="./data_lzj/chunks/",
    contexts_path="./data_lzj/contexts/",
    title_path="./data_lzj/titles.json",
    summaries_path="./data_lzj/summaries.json"
)

In [ ]:
# Clear existing chunk index and rebuild it with all document chunks
elastic_chunk.clear_index()
elastic_chunk.insert_chunks(
    batch_size=1000,
    limit=None,
    skip_existing=True
)

In [ ]:
# Initialize Elasticsearch client for indexing and searching document titles
from lib.search.elastic_title_index import ElasticWriteClientTitles
elastic_title = ElasticWriteClientTitles(
    title_index_name="lzj_titles",
    title_path="./data_lzj/titles.json",
    summaries_path="./data_lzj/summaries.json"
)
#elastic_title.clear_index()
elastic_title.insert_titles(
    limit=None,
    skip_existing=True
)